# Chapter 6: seasonal forecast with a backtest

36 months of one lager. Pull trend and season apart with STL, then hide the last 12 months and forecast them two ways. Synthetic data only; no real brewery's numbers.

What to look at: the MAPE of the seasonal naive forecast against the model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

mv = pd.read_csv('../data/monthly_volume.csv')
print(mv.head(12))

## Trend, season and noise (statsmodels)

STL needs statsmodels: `pip install statsmodels`.

In [ ]:
from statsmodels.tsa.seasonal import STL

res = STL(mv.set_index(pd.to_datetime(mv.month)).volume_hl, period=12).fit()
res.plot()
plt.show()

## Backtest: 24 months to learn, 12 held out

Seasonal naive: last year's same month times last year's growth. The model: a regression on month, trend and the temperature anomaly.

In [ ]:
train, test = mv.iloc[:24], mv.iloc[24:]
growth = train.volume_hl.iloc[12:].sum() / train.volume_hl.iloc[:12].sum()
naive = train.volume_hl.iloc[12:].to_numpy() * growth

def features(d):
    months = pd.get_dummies(d.month_of_year.astype(str)).reindex(columns=[str(m) for m in range(1, 13)], fill_value=0)
    return months.assign(t=d.index.to_numpy(), temp=d.temp_anomaly_c.to_numpy()).astype(float)

reg = LinearRegression().fit(features(train), np.log(train.volume_hl))
ml = np.exp(reg.predict(features(test)))
mape = lambda f: 100 * np.mean(np.abs(test.volume_hl.to_numpy() - f) / test.volume_hl.to_numpy())
print(f'Seasonal naive MAPE {mape(naive):.0f} percent')
print(f'Model MAPE {mape(ml):.0f} percent')

In [ ]:
plt.plot(mv.month, mv.volume_hl, label='actual')
plt.plot(test.month, naive, label='seasonal naive')
plt.plot(test.month, ml, label='model')
plt.xticks(mv.month[::6]); plt.legend(); plt.ylabel('hl')
plt.show()

Two points of MAPE are worth buying only if the error they remove costs more than running the model.